# Pipeline A — Direct VLM prompting (zero-shot)

Shows each temporal collage to **Qwen2.5-VL-7B** and asks: *is there an ephemeral gully in this field?*
Runs on **Missouri** (has ground-truth labels → tells us if our setup works) and **Haryana** (our dataset).

**Before running, in the right-hand panel → Settings:**
1. **Accelerator** → `GPU T4 x2` (or `GPU P100`)
2. **Internet** → On
3. **Add Input** → your dataset `gully-collages`

Then: **Save Version → Save & Run All (Commit)**. You can close the browser; results are saved in the *Output* tab:
`pipeline_a_missouri.csv`, `pipeline_a_haryana.csv`

In [ ]:
# 1. Check the GPU is there
!nvidia-smi

In [ ]:
# 2. Install Ollama (takes ~1 min)
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 3. Start the Ollama server in the background and wait until it answers
import subprocess, time, requests

subprocess.Popen("ollama serve > /kaggle/working/ollama.log 2>&1", shell=True)
for _ in range(60):
    try:
        requests.get("http://localhost:11434", timeout=2)
        print("Ollama server is running")
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Ollama did not start - see /kaggle/working/ollama.log")

In [ ]:
# 4. Download the model (~6 GB, 2-4 min)
!ollama pull qwen2.5vl:7b

In [ ]:
# 5. Settings + find the collages
import base64, csv, glob, json, os
import pandas as pd
from tqdm.auto import tqdm

MODEL = "qwen2.5vl:7b"
OUT_DIR = "/kaggle/working"
MISSOURI_LIMIT = None   # None = all 621. Set e.g. 100 for a quicker run.

def find_dataset(name):
    # Kaggle unzips uploads into slightly different nested folders, so search for the index file
    hits = [p for p in glob.glob("/kaggle/input/**/collages.csv", recursive=True) if name in p.lower()]
    if not hits:
        raise FileNotFoundError(f"No collages.csv for '{name}' under /kaggle/input - did you add the dataset?")
    folder = os.path.dirname(sorted(hits, key=len)[0])
    df = pd.read_csv(folder + "/collages.csv", dtype={"label": str})
    df["path"] = folder + "/" + df["collage"]
    df = df[df["path"].map(os.path.exists)].reset_index(drop=True)
    print(f"{name}: {len(df)} collages in {folder}")
    return df

missouri = find_dataset("missouri")
haryana = find_dataset("haryana")

if MISSOURI_LIMIT:
    # keep the yes/no balance of the full set
    missouri = (missouri.groupby("label")
                .sample(frac=MISSOURI_LIMIT / len(missouri), random_state=42)
                .reset_index(drop=True))
    print("Missouri subset:", len(missouri), missouri["label"].value_counts().to_dict())

In [ ]:
# 6. The question we ask the VLM (identical for both datasets, so the comparison is fair)
PROMPT = '''You are an expert in soil erosion and remote sensing.

The image is a collage of {n} overhead images of the SAME agricultural field, taken at different times.
Each tile has its date or image number written in the top-left corner. Ignore any dark grey "no image"
tile and any light grey text tile - they are not imagery.

An ephemeral gully is a shallow channel carved by concentrated rainwater runoff in cropland. Farmers fill it
during tillage, but it tends to reappear in the same place. In overhead imagery it looks like a narrow,
often slightly curving or branching line - bare or discoloured soil, a darker or lighter strip, or a gap in
the crop - that follows the natural drainage path and usually shows up on several of the dates.
Do NOT count field boundaries, roads, tractor tracks, irrigation channels or straight crop rows.

Question: Is there an ephemeral gully in this field?
First explain what you see in one or two sentences, then answer yes or no.'''

SCHEMA = {
    "type": "object",
    "properties": {
        "reason": {"type": "string"},
        "answer": {"type": "string", "enum": ["yes", "no"]},
    },
    "required": ["reason", "answer"],
}

def ask(image_path, n_images):
    with open(image_path, "rb") as f:
        img = base64.b64encode(f.read()).decode()
    r = requests.post("http://localhost:11434/api/chat", timeout=600, json={
        "model": MODEL,
        "messages": [{"role": "user", "content": PROMPT.format(n=n_images), "images": [img]}],
        "format": SCHEMA,
        "stream": False,
        "options": {"temperature": 0, "num_ctx": 4096, "seed": 42},
    })
    r.raise_for_status()
    raw = r.json()["message"]["content"]
    try:
        out = json.loads(raw)
        answer = str(out.get("answer", "")).strip().lower()
        reason = str(out.get("reason", "")).strip()
    except json.JSONDecodeError:
        answer = "yes" if "yes" in raw.lower()[-20:] else ("no" if "no" in raw.lower()[-20:] else "")
        reason = raw
    pred = 1 if answer == "yes" else (0 if answer == "no" else "")
    return pred, reason, raw

# quick test on one collage from each dataset
for name, df in [("missouri", missouri), ("haryana", haryana)]:
    t = time.time()
    row = df.iloc[0]
    pred, reason, _ = ask(row["path"], row["n_images"])
    print(f"[{name}] tile {row['tile_id']} -> pred={pred}  ({time.time()-t:.1f}s)\n   {reason}\n")

In [ ]:
# 7. Run every collage. Saves after each answer, so a crash loses nothing - just re-run this cell.
def run(name, df):
    out_csv = f"{OUT_DIR}/pipeline_a_{name}.csv"
    done = set(pd.read_csv(out_csv)["tile_id"]) if os.path.exists(out_csv) else set()
    todo = df[~df["tile_id"].isin(done)]
    print(f"{name}: {len(done)} done, {len(todo)} to go")

    new_file = not os.path.exists(out_csv)
    with open(out_csv, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new_file:
            w.writerow(["tile_id", "label", "pred", "reason", "seconds", "raw"])
        for _, row in tqdm(todo.iterrows(), total=len(todo), desc=name):
            t = time.time()
            try:
                pred, reason, raw = ask(row["path"], row["n_images"])
            except Exception as e:
                pred, reason, raw = "", f"ERROR: {e}", ""
            w.writerow([row["tile_id"], row["label"] if pd.notna(row["label"]) else "",
                        pred, reason, round(time.time() - t, 1), raw])
            f.flush()
    return pd.read_csv(out_csv, dtype={"label": str})

res_missouri = run("missouri", missouri)
res_haryana = run("haryana", haryana)

In [ ]:
# 8. Results
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

m = res_missouri.copy()
m = m[m["pred"].astype(str).isin(["0", "1", "0.0", "1.0"]) & m["label"].isin(["0", "1"])]
y_true = m["label"].astype(int)
y_pred = m["pred"].astype(float).astype(int)

print("=== MISSOURI (vs ground truth) ===")
print(f"Scored collages: {len(m)} / {len(res_missouri)}")
print(f"Accuracy : {accuracy_score(y_true, y_pred):.3f}")
print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.3f}")
print(f"Recall   : {recall_score(y_true, y_pred, zero_division=0):.3f}")
print(f"F1       : {f1_score(y_true, y_pred, zero_division=0):.3f}")
print("Confusion matrix [rows = truth 0/1, cols = predicted 0/1]:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))
print("Paper reference (Qwen2-VL-72B, Pipeline A): ~0.71 accuracy")

print("\n=== HARYANA (no labels yet - just what the model said) ===")
print(res_haryana["pred"].value_counts(dropna=False).rename({1: "yes (gully)", 0: "no"}).to_string())
print(f"\nMedian seconds per collage: {pd.concat([res_missouri, res_haryana])['seconds'].median():.1f}")